In [ ]:
import pandas as pd
import numpy as np

#### This is read the gust for expriments in Dashuai et.al

In [ ]:
data = pd.read_csv("naca0012_gust_209.csv")#时间长度264一个case

In [ ]:
inputs_usecols=[]
for i in range(1,10,1):
    #传感器9个
    inputs_usecols.append(f'Cp_{i}')
    

In [ ]:
inputs_usecols

In [ ]:
pressure_csv_data = pd.read_csv("naca0012_gust_209.csv", usecols= inputs_usecols)

In [ ]:
pressure_data_80000 = pressure_csv_data.to_numpy()

# 首先，确保数据的行数可以被121
assert pressure_data_80000.shape[0] % 209 == 0

# 计算每个分组应该有多少行
rows_per_group = pressure_data_80000.shape[0] // 121

# 重塑数组到新的形状 (121, 264, 9)
pressure_data = pressure_data_80000.reshape(209,264,pressure_data_80000.shape[1])

In [ ]:
print("New shape:", pressure_data.shape)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(pressure_data[0,:,1])

#### 拿到力的数据

In [ ]:

outputs_usecols=['CL']
output_csv_data = pd.read_csv("naca0012_gust_209.csv",usecols=outputs_usecols)
output_data_80000 = output_csv_data.to_numpy()
#every 2500 is a case
# 首先，确保数据的行数可以被32整除
assert output_data_80000.shape[0] % 209 == 0


# 计算每个分组应该有多少行
out_per_group = output_data_80000.shape[0] // 209

# 重塑数组到新的形状 (121, 264, 1)
out_data = output_data_80000.reshape(209, out_per_group, 1)

In [ ]:
out_data.shape

In [ ]:
corr = pd.read_csv("foil_coordinates.csv",usecols=["X","Y"])
aoa_x_y = corr.to_numpy()

In [ ]:
# 使用 np.repeat 来沿特定轴重复数据，thanks zijin for point out--2026
coord = aoa_x_y[0:9]

In [ ]:
plt.scatter(coord[0:9,0],coord[0:9,1])

In [ ]:
import torch

In [ ]:

def calculate_distance_matrix(coords):
    num_points = coords.shape[0]
    distance_matrix = np.zeros((num_points, num_points))

    for i in range(num_points):
        for j in range(num_points):
            distance_matrix[i, j] = np.linalg.norm(coords[i] - coords[j])
    return distance_matrix

# 计算连接矩阵
def calculate_adj_matrix(coords, epsilon=1e-5):
    # 计算距离矩阵
    distance_matrix = calculate_distance_matrix(coords)

    # 创建连接矩阵，基于反距离
    adj_matrix = np.zeros_like(distance_matrix)
    for i in range(distance_matrix.shape[0]):
        for j in range(distance_matrix.shape[1]):
            if i != j and distance_matrix[i, j] != 0:
                adj_matrix[i, j] = 1 / (distance_matrix[i, j] + epsilon)

    # 行归一化连接矩阵
    row_sums = adj_matrix.sum(axis=1, keepdims=True)
    adj_matrix_normalized = adj_matrix / (row_sums + epsilon)
    
    return adj_matrix_normalized



In [ ]:
adj = calculate_adj_matrix(coord)

In [ ]:
adj_matrix_batch = np.tile(adj, (209, 1, 1))

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(adj_matrix_batch[0,:,:])

In [ ]:
pressure_data.shape

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch
# 创建一个TensorDataset
# 将NumPy数组转换为PyTorch张量 250长度
pressure_data = torch.tensor(pressure_data[:,14:], dtype=torch.float32)
pressure_coords = torch.tensor(adj_matrix_batch, dtype=torch.float32)
force_output = torch.tensor(out_data[:,14:], dtype=torch.float32)

dataset = TensorDataset(pressure_data, pressure_coords, force_output)
# 确定测试集、验证集的大小
num_total = len(dataset)
num_test = int(num_total * 0.1)
num_val = int((num_total - num_test) * 0.2)
num_train = num_total - num_test - num_val
# 划分数据集
train_dataset, val_dataset, test_dataset = random_split(dataset, [num_train, num_val, num_test], generator=torch.Generator().manual_seed(42))

# 创建DataLoader来批处理数据
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
#检查每个部分的数据大小
print("Training set size:", len(train_dataset))
print("Validation set size:", len(val_dataset))
print("Test set size:", len(test_dataset))

In [ ]:
# 保存train_dataset
dict = {
   "train_dataset":train_dataset,
   "valid_dataset":val_dataset,
   "test_dataset":test_dataset
}
# 保存字典到.pt文件
torch.save(dict, 'simulations_no_augment.pt')